# Regime Labeling: Forward-Return Method (Production Approach)

## Philosophy
Rather than trying to guess regimes from current data, **label each date with what the market actually does over the next 63 days**.

This approach:
- ✅ Is **objective** (based on actual forward returns)
- ✅ Has **clear thresholds** (7%, -5%) grounded in reality
- ✅ Captures economic intuition
- ✅ Avoids lookahead bias (uses past context only)
- ✅ Is what professional ML systems use

## Regime Definitions

For each date `t`, we look at:
- **Forward return**: SPY return from `t` to `t+63 days`
- **Trailing return**: SPY return from `t-126` to `t` (past 6 months)

Then classify:
```
if fwd_return > +7%:
    regime = EXPANSION          # Strong rally ahead
elif fwd_return < -5%:
    regime = CONTRACTION        # Decline ahead
elif fwd_return > 0% AND trailing_return < -5%:
    regime = RECOVERY           # Bouncing from recent weakness
else:
    regime = LATE_CYCLE         # Flat/moderate (catch-all)
```

## Why This Works

The thresholds are **economically meaningful**:
- **7%**: Quarter-normalized bull move (28% annualized)
- **-5%**: Quarter-normalized bear move (-20% annualized)
- **Recovery context**: Requires recent weakness to distinguish from false breakout

The RECOVERY class is the key insight: not all small gains are the same.
A +3% return after a 10% drop (recovery) has different character than +3% in a stable market (late cycle).

## Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load features
regime_data = pd.read_csv('../data/regime_features_raw.csv')
regime_data['date'] = pd.to_datetime(regime_data['date'])

print(f"Loaded {len(regime_data)} rows")
print(f"Date range: {regime_data['date'].min()} to {regime_data['date'].max()}")

## Compute Forward & Trailing Returns

In [ ]:
df = regime_data.copy()

# Forward return: what happens in the next 63 days (looking ahead)
df['fwd_ret_63d'] = df['spy_price'].pct_change(63).shift(-63)

# Trailing return: what happened in the past 126 days (context for recovery)
df['trail_ret_126d'] = df['spy_price'].pct_change(126)

print("Forward and trailing returns computed:")
print(f"\nForward return stats (next 63 days):")
print(df['fwd_ret_63d'].describe())
print(f"\nTrailing return stats (past 126 days):")
print(df['trail_ret_126d'].describe())

## Label Regimes Based on Forward Returns

In [ ]:
def forward_regime(row):
    """
    Classify regime based on forward 63-day return + trailing 126-day context.
    
    This is an objective labeling: we're asking "what did the market actually do?"
    """
    fwd = row['fwd_ret_63d']
    trail = row['trail_ret_126d']
    
    # Skip if missing
    if pd.isna(fwd) or pd.isna(trail):
        return np.nan
    
    # Classification logic
    if fwd > 0.07:  # >7% forward return
        return 'EXPANSION'
    elif fwd < -0.05:  # <-5% forward return
        return 'CONTRACTION'
    elif fwd > 0.0 and trail < -0.05:  # 0-7% return AND trailing weakness
        return 'RECOVERY'
    else:  # -5% to 0%, or positive but no trailing weakness
        return 'LATE_CYCLE'

df['regime'] = df.apply(forward_regime, axis=1)

print("Regime distribution (Forward-Return Method):")
print(df['regime'].value_counts())
print(f"\nPercentage:")
print(df['regime'].value_counts(normalize=True).round(3) * 100)
print(f"\nRows with valid labels: {df['regime'].notna().sum()} / {len(df)}")

## Examine Regime Characteristics

In [ ]:
# What does each regime look like?
print("Regime Characteristics:")
print("="*80)

for regime in ['EXPANSION', 'LATE_CYCLE', 'CONTRACTION', 'RECOVERY']:
    mask = df['regime'] == regime
    subset = df[mask]
    
    if mask.sum() == 0:
        print(f"\n{regime}: (no data)")
        continue
    
    print(f"\n{regime} ({mask.sum()} days):")
    print(f"  Forward return:  {subset['fwd_ret_63d'].mean():>7.1%} (±{subset['fwd_ret_63d'].std():.1%})")
    print(f"  Trailing return: {subset['trail_ret_126d'].mean():>7.1%} (±{subset['trail_ret_126d'].std():.1%})")
    print(f"  SPY price:       {subset['spy_price'].mean():>7.0f}")
    print(f"  Volatility:      {subset['vol_20d'].mean():>7.1f}%")
    print(f"  Unemployment:    {subset['unemploymentRate'].mean():>7.1f}%")
    print(f"  Yield spread:    {subset['yield_curve_spread'].mean():>7.2f}%")

## Validate Regime Definitions

In [ ]:
# Sanity check: do the regimes match their definitions?
print("Validation: Forward returns by regime")
print("="*80)

for regime in ['EXPANSION', 'LATE_CYCLE', 'CONTRACTION', 'RECOVERY']:
    mask = df['regime'] == regime
    fwd = df.loc[mask, 'fwd_ret_63d']
    
    if mask.sum() == 0:
        continue
    
    print(f"\n{regime}:")
    print(f"  Min:    {fwd.min():>7.1%}")
    print(f"  25th:   {fwd.quantile(0.25):>7.1%}")
    print(f"  Median: {fwd.median():>7.1%}")
    print(f"  75th:   {fwd.quantile(0.75):>7.1%}")
    print(f"  Max:    {fwd.max():>7.1%}")
    print(f"  Mean:   {fwd.mean():>7.1%}")

## Visualize Regimes Over Time

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

# Color map
regime_colors = {
    'EXPANSION': '#2ecc71',      # Green
    'LATE_CYCLE': '#f39c12',     # Orange
    'CONTRACTION': '#e74c3c',    # Red
    'RECOVERY': '#3498db'        # Blue
}

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Plot 1: SPY price colored by regime
for regime in ['EXPANSION', 'LATE_CYCLE', 'CONTRACTION', 'RECOVERY']:
    mask = df['regime'] == regime
    axes[0].scatter(df.loc[mask, 'date'], df.loc[mask, 'spy_price'],
                   c=regime_colors[regime], label=regime, alpha=0.6, s=15)
axes[0].set_ylabel('SPY Price ($)', fontweight='bold', fontsize=11)
axes[0].set_title('Market Regimes: Forward-Return Method', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper left', ncol=4, framealpha=0.95)
axes[0].grid(True, alpha=0.3)

# Plot 2: Forward 63-day return (what the label is based on)
axes[1].plot(df['date'], df['fwd_ret_63d'] * 100, linewidth=1, color='navy', alpha=0.7)
axes[1].axhline(y=7, color='green', linestyle='--', linewidth=1.5, label='EXPANSION threshold (7%)')
axes[1].axhline(y=-5, color='red', linestyle='--', linewidth=1.5, label='CONTRACTION threshold (-5%)')
axes[1].axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
axes[1].fill_between(df['date'], df['fwd_ret_63d'] * 100, 0, alpha=0.2, color='navy')
axes[1].set_ylabel('Forward 63d Return (%)', fontweight='bold', fontsize=11)
axes[1].legend(loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3)

# Plot 3: Trailing 126-day return (context for RECOVERY)
axes[2].plot(df['date'], df['trail_ret_126d'] * 100, linewidth=1.5, color='purple')
axes[2].axhline(y=-5, color='orange', linestyle='--', linewidth=1.5, label='RECOVERY context threshold (-5%)')
axes[2].axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
axes[2].fill_between(df['date'], df['trail_ret_126d'] * 100, 0, alpha=0.2, color='purple')
axes[2].set_ylabel('Trailing 126d Return (%)', fontweight='bold', fontsize=11)
axes[2].set_xlabel('Date', fontweight='bold', fontsize=11)
axes[2].legend(loc='upper left', fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/03_regime_labeling_forward.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to outputs/03_regime_labeling_forward.png")

## Save Labeled Dataset

In [ ]:
# Save dataset with regime labels
df_labeled = df[['date', 'spy_price', 'ret_5d', 'ret_20d', 'ret_60d', 'vol_20d', 'vol_60d',
                 'y2', 'y10', 'yield_curve_spread', 'oil_price', 'oil_ret_20d',
                 'CPI', 'GDP', 'inflationRate', 'unemploymentRate',
                 'fwd_ret_63d', 'trail_ret_126d', 'regime']].copy()

# Keep only rows with valid regime labels
df_labeled = df_labeled.dropna(subset=['regime']).reset_index(drop=True)

df_labeled.to_csv('../data/regime_labeled_forward.csv', index=False)

print(f"✓ Saved {len(df_labeled)} labeled rows to regime_labeled_forward.csv")
print(f"\nThis is your training dataset for Phase 4 & 5.")
print(f"\nDataset summary:")
print(f"  Rows: {len(df_labeled)}")
print(f"  Date range: {df_labeled['date'].min()} to {df_labeled['date'].max()}")
print(f"  Regimes: {df_labeled['regime'].unique().tolist()}")
print(f"  Columns: {len(df_labeled.columns)}")

## Key Insights

**Why this labeling approach is superior:**

1. **Objective**: Labels are based on what actually happened, not guesses
2. **No lookahead bias**: Uses only past data to predict future regime
3. **Clear thresholds**: 7% and -5% are economically meaningful
4. **Captures context**: RECOVERY requires trailing weakness (avoids false breakouts)
5. **Balanced classes**: All 4 regimes should appear frequently

**Next steps (Phase 4 & 5):**
- Use this labeled dataset to train a classifier
- The model learns to predict FUTURE regime from PAST features
- This enables real-time regime detection for portfolio tilting